<a href="https://colab.research.google.com/github/AnanyaAsthana/Data-Mining-Warehousing-Lab/blob/main/Lab8_dwm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# EXPERIMENT 5: SIMILARITY AND DISSIMILARITY MEASURES

import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. CREATE SAMPLE DATASETS
# ------------------------------------------------

# Sample Patients Dataset (Binary Attributes)
patients = pd.DataFrame({
    'Patient_ID': [1, 2, 3, 4, 5],
    'Fever': [1, 1, 0, 1, 0],
    'Cough': [1, 0, 1, 1, 0],
    'Fatigue': [0, 1, 1, 1, 0],
    'Headache': [1, 0, 0, 1, 1]
})

# Sample Customers Dataset (Numerical + Binary + Nominal)
customers = pd.DataFrame({
    'Customer_ID': [1, 2, 3, 4, 5],
    'Age': [22, 35, 28, 45, 30],
    'Income': [25000, 50000, 35000, 70000, 40000],
    'Spending_Score': [80, 60, 75, 40, 65],
    'Gender': ['Male', 'Female', 'Male', 'Female', 'Male'],
    'Membership': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver'],
    'Premium': [1, 0, 1, 0, 1]
})

print("PATIENTS DATASET")
print(patients)

print("\nCUSTOMERS DATASET")
print(customers)


# ------------------------------------------------
# 2. SIMPLE MATCHING COEFFICIENT AND JACCARD
# ------------------------------------------------

binary_cols = ['Fever', 'Cough', 'Fatigue', 'Headache']

binary_data = patients[binary_cols]

A = binary_data.iloc[0].values
B = binary_data.iloc[1].values

# Simple Matching Coefficient
matching = np.sum(A == B)
SMC = matching / len(A)

# Jaccard Similarity
intersection = np.sum((A == 1) & (B == 1))
union = np.sum((A == 1) | (B == 1))

Jaccard = intersection / union if union != 0 else 0

print("\nSIMPLE MATCHING COEFFICIENT:", SMC)
print("JACCARD SIMILARITY:", Jaccard)


# ------------------------------------------------
# 3. BINARY SIMILARITY MATRIX
# ------------------------------------------------

def jaccard_similarity(a, b):
    intersection = np.sum((a == 1) & (b == 1))
    union = np.sum((a == 1) | (b == 1))

    return intersection / union if union != 0 else 0


def smc_similarity(a, b):
    return np.mean(a == b)


n = len(binary_data)

jaccard_matrix = np.zeros((n, n))
smc_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):

        a = binary_data.iloc[i].values
        b = binary_data.iloc[j].values

        jaccard_matrix[i, j] = jaccard_similarity(a, b)
        smc_matrix[i, j] = smc_similarity(a, b)


print("\nJACCARD SIMILARITY MATRIX")
print(pd.DataFrame(jaccard_matrix).round(3))

print("\nSIMPLE MATCHING COEFFICIENT MATRIX")
print(pd.DataFrame(smc_matrix).round(3))


# ------------------------------------------------
# 4. NORMALIZED NUMERICAL DISSIMILARITY
# ------------------------------------------------

numeric_cols = ['Age', 'Income', 'Spending_Score']

numeric_data = customers[numeric_cols]

# Min-Max Normalization
normalized = (
    (numeric_data - numeric_data.min()) /
    (numeric_data.max() - numeric_data.min())
)

print("\nNORMALIZED NUMERICAL DATA")
print(normalized.round(3))

# Calculate normalized Manhattan dissimilarity
n = len(normalized)

numerical_dissimilarity = np.zeros((n, n))

for i in range(n):
    for j in range(n):

        numerical_dissimilarity[i, j] = np.mean(
            np.abs(
                normalized.iloc[i].values -
                normalized.iloc[j].values
            )
        )


print("\nNORMALIZED NUMERICAL DISSIMILARITY MATRIX")
print(pd.DataFrame(numerical_dissimilarity).round(3))


# ------------------------------------------------
# 5. MIXED ATTRIBUTE DISSIMILARITY MATRIX
# ------------------------------------------------

mixed_data = customers[
    ['Age', 'Income', 'Spending_Score',
     'Gender', 'Membership', 'Premium']
].copy()

num_cols = ['Age', 'Income', 'Spending_Score']

bin_cols = ['Premium']

nom_cols = ['Gender', 'Membership']


def gower_dissimilarity(a, b, data):

    distances = []

    for col in data.columns:

        x = a[col]
        y = b[col]

        if col in num_cols:

            value_range = data[col].max() - data[col].min()

            if value_range == 0:
                d = 0
            else:
                d = abs(x - y) / value_range

        elif col in bin_cols:

            d = 0 if x == y else 1

        else:

            d = 0 if x == y else 1

        distances.append(d)

    return np.mean(distances)


n = len(mixed_data)

mixed_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):

        mixed_matrix[i, j] = gower_dissimilarity(
            mixed_data.iloc[i],
            mixed_data.iloc[j],
            mixed_data
        )


print("\nMIXED ATTRIBUTE DISSIMILARITY MATRIX")
print(pd.DataFrame(mixed_matrix).round(3))


# ------------------------------------------------
# 6. CONCLUSION
# ------------------------------------------------

print("\nCONCLUSION:")

print("1. SMC measures matching binary attributes.")
print("2. Jaccard measures similarity of positive attributes.")
print("3. Numerical dissimilarity measures normalized differences.")
print("4. Gower dissimilarity handles numerical, binary and nominal attributes.")

print("\nEXPERIMENT COMPLETED SUCCESSFULLY!")


PATIENTS DATASET
   Patient_ID  Fever  Cough  Fatigue  Headache
0           1      1      1        0         1
1           2      1      0        1         0
2           3      0      1        1         0
3           4      1      1        1         1
4           5      0      0        0         1

CUSTOMERS DATASET
   Customer_ID  Age  Income  Spending_Score  Gender Membership  Premium
0            1   22   25000              80    Male       Gold        1
1            2   35   50000              60  Female     Silver        0
2            3   28   35000              75    Male       Gold        1
3            4   45   70000              40  Female     Bronze        0
4            5   30   40000              65    Male     Silver        1

SIMPLE MATCHING COEFFICIENT: 0.25
JACCARD SIMILARITY: 0.25

JACCARD SIMILARITY MATRIX
       0      1      2     3      4
0  1.000  0.250  0.250  0.75  0.333
1  0.250  1.000  0.333  0.50  0.000
2  0.250  0.333  1.000  0.50  0.000
3  0.750  0.500  0.